# Etapa 2: Pré-Processamento, Encodings e Escalonamento Numérico

## 1. Contexto e Objetivo

Este notebook realiza o pipeline de pré-processamento estrito e anti-vazamento (*Data Leakage Free*):
1. **Split Temporal:**
   * **Treino:** 01/01/2023 a 31/12/2025 (523.888 registros operacionais).
   * **Teste Cego:** 01/01/2026 a 28/02/2026 (28.202 registros operacionais).
2. **Encodings Categóricos:**
   * One-Hot Encoding nas 10 estações monitoradas.
   * Binary Mapping na variável `plataforma` (`P1` = 0, `P2` = 1).
3. **Escalonamento Numérico:**
   * `RobustScaler` ajustado (*fit*) **exclusivamente no Treino** e aplicado (*transform*) no Teste.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
print("✅ Bibliotecas carregadas com sucesso!")


✅ Bibliotecas carregadas com sucesso!


## 2. Carregamento dos Dados e Divisão Temporal


In [2]:
path_mestra = "../../data/tabela_mestra_bruta.csv"
df = pd.read_csv(path_mestra)
df['data_ref'] = pd.to_datetime(df['data_ref'])

# Features 2.0 (se ainda não calculadas)
if 'aceleracao_alarmes_7d' not in df.columns:
    df['aceleracao_alarmes_7d'] = df['qtd_alarmes_7d'] - (df['qtd_alarmes_14d'] - df['qtd_alarmes_7d'])
if 'tempo_medio_por_alarme_7d' not in df.columns:
    df['tempo_medio_por_alarme_7d'] = df['tempo_parada_acumulado_7d'] / (df['qtd_alarmes_7d'] + 1)
if 'proporcao_alarmes_graves_7d' not in df.columns:
    df['proporcao_alarmes_graves_7d'] = df['qtd_anomalias_30s_14d'] / (df['qtd_alarmes_14d'] + 1)
if 'faixa_risco_zm01' not in df.columns:
    df['faixa_risco_zm01'] = np.where((df['dias_desde_ultima_zm01'] >= 30) & (df['dias_desde_ultima_zm01'] <= 90), 1, 0)

data_corte = pd.to_datetime('2026-01-01')
df_treino = df[df['data_ref'] < data_corte].copy()
df_teste = df[df['data_ref'] >= data_corte].copy()

print(f"Treino (2023-2025): {df_treino.shape[0]:,} linhas")
print(f"Teste (2026):       {df_teste.shape[0]:,} linhas")



Treino (2023-2025): 523,888 linhas
Teste (2026):       28,202 linhas


## 3. Aplicação de Encodings Categóricos


In [3]:
def aplicar_encodings(df_in):
    df_out = df_in.copy()
    df_out['plataforma_bin'] = df_out['plataforma'].map({'P1': 0, 'P2': 1}).fillna(0).astype(int)
    df_out = pd.get_dummies(df_out, columns=['estacao'], prefix='estacao', drop_first=False)
    return df_out

df_treino_enc = aplicar_encodings(df_treino)
df_teste_enc = aplicar_encodings(df_teste)
df_treino_enc, df_teste_enc = df_treino_enc.align(df_teste_enc, join='left', axis=1, fill_value=0)

print(f"Colunas após Encodings: {df_treino_enc.shape[1]}")



Colunas após Encodings: 37


## 4. Escalonamento Numérico e Exportação dos Datasets Finais


In [4]:
cols_para_escalonar = [
    'idade_ativo_anos', 'porta',
    'dias_desde_ultima_zm01', 'dias_desde_ultima_zm02', 'dias_desde_ultima_zm03',
    'qtd_alarmes_1d', 'qtd_alarmes_7d', 'qtd_alarmes_14d',
    'aceleracao_alarmes_7d', 'qtd_anomalias_30s_14d',
    'tempo_parada_acumulado_7d', 'max_tempo_normalizacao_7d',
    'tempo_medio_por_alarme_7d', 'proporcao_alarmes_graves_7d'
]

scaler = RobustScaler()
scaler.fit(df_treino_enc[cols_para_escalonar])

cols_scaled = [f"{c}_scaled" for c in cols_para_escalonar]
df_treino_enc[cols_scaled] = scaler.transform(df_treino_enc[cols_para_escalonar])
df_teste_enc[cols_scaled] = scaler.transform(df_teste_enc[cols_para_escalonar])

# Salvar Scaler
os.makedirs("../../data/modelos_artefatos", exist_ok=True)
joblib.dump(scaler, "../../data/modelos_artefatos/robust_scaler.joblib")
print("✅ Scaler ajustado e salvo em ../../data/modelos_artefatos/robust_scaler.joblib!")

cols_estacoes = [c for c in df_treino_enc.columns if c.startswith('estacao_')]
features_finais = ['plataforma_bin', 'faixa_risco_zm01'] + cols_estacoes + cols_scaled
target_col = 'target_falha_7d'
cols_export = ['id_ativo', 'data_ref'] + features_finais + [target_col]

path_treino_out = "../../data/treino_pronto_modelagem_v2.csv"
path_teste_out = "../../data/teste_pronto_modelagem_v2.csv"

df_treino_enc[cols_export].to_csv(path_treino_out, index=False)
df_teste_enc[cols_export].to_csv(path_teste_out, index=False)

print(f"\n✅ Datasets finais de modelagem exportados com sucesso!")
print(f"Treino: {path_treino_out} ({df_treino_enc.shape[0]:,} linhas, {len(cols_export)} colunas)")
print(f"Teste:  {path_teste_out} ({df_teste_enc.shape[0]:,} linhas, {len(cols_export)} colunas)")



✅ Scaler ajustado e salvo em ../../data/modelos_artefatos/robust_scaler.joblib!

✅ Datasets finais de modelagem exportados com sucesso!
Treino: ../../data/treino_pronto_modelagem_v2.csv (523,888 linhas, 29 colunas)
Teste:  ../../data/teste_pronto_modelagem_v2.csv (28,202 linhas, 29 colunas)
